In [1]:
import pandas as pd
from data import BasinDeltaTable

In [2]:
store = BasinDeltaTable(root_dir="./my_delta_dataset", overwrite=True)

Overwriting. Deleting directory: my_delta_dataset


In [3]:
# 2. Append data for multiple subbasins (this creates small files)
for i in range(5):
    dates = pd.to_datetime(pd.date_range("2021-01-01", "2021-01-10", freq="D"))
    data = pd.DataFrame(index=dates, data={'flow': range(10)})
    store.upsert_dynamic(
        basin='mississippi',
        subbasin=f'sub_{i}',
        source='ERA5',
        data=data
    )

Table not found at my_delta_dataset/dynamic_data. Creating new Delta table.


In [6]:
store.is_processed('mississippi','sub_0','ERA5')

True

In [14]:
store.get_processing_status('mississippi','ERA5')

,basin,subbasin,source,processed_at,has_data
0,mississippi,sub_4,ERA5,2025-09-17 19:23:32.881140+00:00,True
1,mississippi,sub_3,ERA5,2025-09-17 19:23:32.787286+00:00,True
2,mississippi,sub_2,ERA5,2025-09-17 19:23:32.487589+00:00,True
3,mississippi,sub_1,ERA5,2025-09-17 19:23:32.019451+00:00,True
4,mississippi,sub_0,ERA5,2025-09-17 19:23:31.232610+00:00,True


In [9]:
# 3. Read the data - it all works seamlessly
filtered_df = store.read_dynamic(basin='mississippi', subbasins=['sub_1'])
filtered_df

,flow,year,basin,subbasin,source
date,,,,,
2021-01-01 00:00:00+00:00,0,2021,mississippi,sub_1,ERA5
2021-01-02 00:00:00+00:00,1,2021,mississippi,sub_1,ERA5
2021-01-03 00:00:00+00:00,2,2021,mississippi,sub_1,ERA5
2021-01-04 00:00:00+00:00,3,2021,mississippi,sub_1,ERA5
2021-01-05 00:00:00+00:00,4,2021,mississippi,sub_1,ERA5
2021-01-06 00:00:00+00:00,5,2021,mississippi,sub_1,ERA5
2021-01-07 00:00:00+00:00,6,2021,mississippi,sub_1,ERA5
2021-01-08 00:00:00+00:00,7,2021,mississippi,sub_1,ERA5
2021-01-09 00:00:00+00:00,8,2021,mississippi,sub_1,ERA5


In [11]:
store.compact()

Compacting dynamic_data...
Files added: 0, Files removed: 0.
Compacting processing metadata...
Files added: 0, Files removed: 0.


In [12]:
store.vacuum(retention_hours=0) 

Starting vacuum of dynamic data...
Vacuum complete. Files deleted: 7
Starting vacuum of processing metadata...
Vacuum complete. Files deleted: 5


In [13]:
store.table_uri

'my_delta_dataset/dynamic_data'